# NfgTransformer SRE Solver Training

This notebook trains the neural SRE stage-game solver used by Deep SRQ. The default workflow samples fresh synthetic games online and optimizes robust exploitability directly.

1. Train the NfgTransformer with epsilon-conditioned robust-exploitability loss.
2. Generate held-out random normal-form games.
3. Evaluate a checkpoint.
4. Plug the checkpoint into Level-Based Foraging Deep SRQ.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'SRE-DQN' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / 'nfg_sre_data'
CKPT_ROOT = REPO_ROOT / 'nfg_sre_checkpoints'

GAME_SHAPES = '6x6x6,4x5x6'
EVAL_GAME_SHAPE = '6x6x6'
VAL_DIR = DATA_ROOT / 'val_lbf3_random'
CHECKPOINT = CKPT_ROOT / 'nfg_sre_lbf3_online.pt'

VAL_SAMPLES = 64
SHARD_SIZE = 64

DATA_ROOT.mkdir(exist_ok=True)
CKPT_ROOT.mkdir(exist_ok=True)
print(REPO_ROOT)

/home/wowthecoder/SRE-DQN


## 1. Train the NfgTransformer checkpoint

In [2]:
from discrete_action_space.sre_solvers.nfg_transformer.train import train_checkpoint

train_checkpoint(
    output=CHECKPOINT,
    epochs=5,
    batches_per_epoch=100,
    batch_size=64,
    lr=3e-4,
    embed_dim=64,
    num_blocks=8,
    num_heads=8,
    num_self_attend_per_block=1,
    game_shapes=GAME_SHAPES,
    seed=2025,
    use_gpu=True,
)

training_mode=online_synthetic_robust_gap shapes=[(6, 6, 6), (4, 5, 6)] batch_size=64 device=cuda


nfg-sre-train:1: 100%|████████████████████████████████████████████████████████████████| 100/100 [02:25<00:00,  1.45s/it]


epoch=1 loss=0.331248 robust_gap=0.331248


nfg-sre-train:2: 100%|████████████████████████████████████████████████████████████████| 100/100 [02:17<00:00,  1.37s/it]


epoch=2 loss=0.202778 robust_gap=0.202778


...

epoch=3 loss=0.157759 robust_gap=0.157759


nfg-sre-train:4: 100%|████████████████████████████████████████████████████████████████| 100/100 [02:15<00:00,  1.35s/it]


epoch=4 loss=0.128784 robust_gap=0.128784


nfg-sre-train:5: 100%|████████████████████████████████████████████████████████████████| 100/100 [02:14<00:00,  1.35s/it]

epoch=5 loss=0.120142 robust_gap=0.120142


## 2. Generate held-out evaluation data

Evaluation shards use one concrete tensor shape because NumPy arrays cannot mix rectangular game sizes inside one shard.

In [3]:
from discrete_action_space.sre_solvers.nfg_transformer.generate_dataset import generate_dataset

generate_dataset(
    output=VAL_DIR,
    num_samples=VAL_SAMPLES,
    shard_size=SHARD_SIZE,
    num_players=3,
    num_actions=6,
    game_shape=EVAL_GAME_SHAPE,
    seed=2026,
    label_mode='random',
    exploitability_tol=1e-4,
)

nfg-sre-random: 100%|█████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 5122.03it/s]


## 3. Evaluate the checkpoint

The main number is robust exploitability. A low accept rate is okay for early checkpoints because the integrated solver falls back to PATH.

In [4]:
from discrete_action_space.sre_solvers.nfg_transformer.evaluate import evaluate_checkpoint

evaluate_checkpoint(
    checkpoint=CHECKPOINT,
    data_dir=VAL_DIR,
    exploitability_tol=1e-3,
    device=None,
)

checkpoint=/home/wowthecoder/SRE-DQN/nfg_sre_checkpoints/nfg_sre_lbf3_online.pt
samples=64
mean_gap=0.119200
p95_gap=0.212648
accept_rate=0.0000


## 4. Use in Level-Based Foraging Deep SRQ

This uses the neural solver first. If the neural robust gap is above `nfg_accept_gap`, it falls back to PATH MCP.

In [ ]:
from discrete_action_space.lbf_grid.deep_srq_lbf import train_lbf_deep_srq_experiment

# Keep this small for a smoke run; increase n_episodes for real experiments.
stats = train_lbf_deep_srq_experiment(
    n_episodes=20,
    solver_name='nfg_transformer_sre',
    epsilon_robust_initial=0.5,
    epsilon_schedule='linear',
    hyperparameter_overrides={
        'nfg_checkpoint_path': str(CHECKPOINT),
        'nfg_accept_gap': 1e-3,
        'nfg_fallback_enabled': True,
    },
    write_plots=False,
    print_full_stats=True,
)

stats['timing']

LBF DeepSRQ | players=3 | solver=nfg_transformer_sre | eps0=0.5 | schedule=linear | seed=2025


lbf:nfg_transformer_sre_eps0.5_linear:  45%|████████████████████▎                        | 9/20 [01:41<03:06, 16.98s/it]